In [6]:
import pandas as pd
# import scanpy as sc
import numpy as np
# import anndata as ad
import json

from choose_protein_coding import list_of_protein_coding_genes

In [7]:
# configure
filename = "TCGA-BRCA.star_counts"

# for MLP

In [28]:
df_original = pd.read_csv(f'../data/raw_tsv_data/{filename}.tsv', delimiter='\t', index_col=False)
df = df_original.rename(columns={'Ensembl_ID': 'Unnamed: 0-1'})

In [29]:
df.head()

,Unnamed: 0-1,TCGA-D8-A146-01A,TCGA-AQ-A0Y5-01A,TCGA-C8-A274-01A,TCGA-BH-A0BD-01A,TCGA-B6-A1KC-01B,TCGA-AC-A62V-01A,TCGA-AO-A0J5-01A,TCGA-BH-A0B1-01A,TCGA-A2-A0YM-01A,...,TCGA-E2-A1IG-01A,TCGA-E9-A1NA-01A,TCGA-D8-A1JP-01A,TCGA-AR-A252-01A,TCGA-D8-A1XL-01A,TCGA-BH-A0EI-01A,TCGA-E2-A1IO-01A,TCGA-E2-A15R-01A,TCGA-B6-A0IP-01A,TCGA-A1-A0SN-01A
0,ENSG00000000003.15,11.737670,9.781360,13.122504,11.016808,11.000000,9.614710,11.092096,12.103616,12.417325,...,10.747354,10.360847,10.675957,11.337064,11.697836,12.339850,12.131857,8.974415,13.326289,9.710806
1,ENSG00000000005.6,7.721099,3.321928,0.000000,6.686501,3.807355,4.087463,5.392317,3.000000,3.321928,...,1.584963,2.000000,1.000000,8.164907,0.000000,2.807355,3.906891,3.700440,3.807355,1.000000
2,ENSG00000000419.13,11.042343,11.357552,11.506308,10.801708,11.074141,11.107217,10.318543,11.160502,11.072803,...,10.429407,10.811375,10.835261,10.693487,11.922213,11.221587,10.913637,11.659550,10.962173,12.416270
3,ENSG00000000457.14,11.036860,10.754888,12.218260,11.190442,10.857981,8.539159,11.527966,10.635718,9.945444,...,10.366322,10.558421,11.262682,10.751544,10.607330,10.163650,10.575539,11.929258,11.403012,10.870365
4,ENSG00000000460.17,9.131857,8.721099,10.973697,10.761551,9.550747,8.550747,9.177420,9.727920,10.504819,...,8.661778,9.000000,9.794416,9.231221,9.782998,9.252665,8.632995,10.562242,10.060696,9.722808


In [30]:
# choose 01A for samples with duplicates

col_df = pd.DataFrame({"col": df.columns})

# prefix bez ostatniej litery
col_df["prefix"] = col_df["col"].str[:-1]

# suffix = ostatnia litera
col_df["suffix"] = col_df["col"].str[-1]

col_df["is_A"] = (col_df["suffix"] == "A").astype(int)
col_df = col_df.sort_values(
    ["prefix", "is_A"],
    ascending=[True, False]
)
selected_cols = col_df.drop_duplicates("prefix")["col"].tolist()
selected_cols.insert(0, 'Unnamed: 0-1')


In [31]:
df = df[selected_cols]
df.columns = df.columns.str.split("-").str[:-1].str.join("-")

df = df.T
df.columns = df.iloc[0]   # pierwszy wiersz → nazwy kolumn
df = df.iloc[1:]

In [32]:
df.head()

Unnamed: 0,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288661.1,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1
TCGA-3C-AAAU,9.348728,1.584963,10.871135,9.894818,8.409391,7.960002,9.83289,10.14083,10.632995,11.756139,...,0.0,0.0,5.491853,0.0,0.0,0.0,9.670656,0.0,4.321928,5.209453
TCGA-3C-AALI,8.710806,1.584963,10.82893,11.251482,8.939579,8.603626,10.778077,10.510764,10.663558,10.291171,...,0.0,1.0,5.584963,0.0,0.0,1.0,8.118941,0.0,3.321928,5.523562
TCGA-3C-AALJ,10.348728,5.72792,10.324181,8.573647,7.906891,8.539159,9.926296,9.584963,9.422065,9.5157,...,0.0,1.0,5.169925,0.0,0.0,0.0,7.219169,0.0,2.584963,4.807355
TCGA-3C-AALK,11.536247,2.321928,10.437752,10.361944,8.848623,8.588715,11.384244,10.580259,10.504819,11.316847,...,0.0,0.0,5.491853,0.0,1.0,0.0,8.118941,0.0,3.906891,5.78136
TCGA-4H-AAAK,11.329236,4.0,10.491853,9.97871,8.519636,8.47978,11.628901,10.690871,10.520619,10.673309,...,0.0,0.0,5.169925,0.0,0.0,0.0,7.982994,0.0,2.807355,5.209453


In [33]:
df.columns = df.columns.str.split(".").str[0]

features = pd.read_csv("../data/gene_info.csv")
features = features[["feature_id", "feature_name"]]

id_to_symbol = dict(
    zip(features["feature_id"], features["feature_name"])
)

df = df.rename(columns=id_to_symbol)

df = df.loc[:, df.columns.notnull()]
df = df.loc[:, ~df.columns.duplicated()]

df.shape

(1216, 60616)

In [34]:
df.head()

Unnamed: 0,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,RP11-357C3.5,RP11-73F12.1,RP11-680A11.7,RP11-439H13.3,RP11-73F12.2,CTD-3214H19.17,RP11-122G18.14,AC006486.14,RP5-1087E8.6,PANO1
TCGA-3C-AAAU,9.348728,1.584963,10.871135,9.894818,8.409391,7.960002,9.83289,10.14083,10.632995,11.756139,...,0.0,0.0,5.491853,0.0,0.0,0.0,9.670656,0.0,4.321928,5.209453
TCGA-3C-AALI,8.710806,1.584963,10.82893,11.251482,8.939579,8.603626,10.778077,10.510764,10.663558,10.291171,...,0.0,1.0,5.584963,0.0,0.0,1.0,8.118941,0.0,3.321928,5.523562
TCGA-3C-AALJ,10.348728,5.72792,10.324181,8.573647,7.906891,8.539159,9.926296,9.584963,9.422065,9.5157,...,0.0,1.0,5.169925,0.0,0.0,0.0,7.219169,0.0,2.584963,4.807355
TCGA-3C-AALK,11.536247,2.321928,10.437752,10.361944,8.848623,8.588715,11.384244,10.580259,10.504819,11.316847,...,0.0,0.0,5.491853,0.0,1.0,0.0,8.118941,0.0,3.906891,5.78136
TCGA-4H-AAAK,11.329236,4.0,10.491853,9.97871,8.519636,8.47978,11.628901,10.690871,10.520619,10.673309,...,0.0,0.0,5.169925,0.0,0.0,0.0,7.982994,0.0,2.807355,5.209453


In [35]:
# keep only protein-coding genes

gene_list = df.columns
gene_list = gene_list[1:]
protein_coding_gene_list = list_of_protein_coding_genes(gene_list)

df_filtered = df[ df.columns.intersection(protein_coding_gene_list) ]


In [36]:
df_filtered = df_filtered[~df_filtered.index.duplicated(keep="first")]


In [51]:
# save to file
df_filtered.to_csv(f"../data/0_data_for_mlp/{filename}.csv")


# for scGPT

In [52]:
# create anndata df

X = df_filtered.values.astype(np.float32)

obs = pd.DataFrame(index=df_filtered.index)
obs["sample"] = df_filtered.index

var = pd.DataFrame(index=df_filtered.columns)
var["gene_name"] = df_filtered.columns

adata = ad.AnnData(
    X=X,
    obs=obs,
    var=var
)

adata

AnnData object with n_obs × n_vars = 1095 × 20260
    obs: 'sample'
    var: 'gene_name'

In [53]:
adata.write(f"../data/0_adata_for_scgpt/adata_{filename}.h5ad")
